In [ ]:
import warnings 
warnings.filterwarnings('ignore')
# Load Data :

from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf')
pages = loader.load()

# Build Chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=700 , chunk_overlap = 75)
text = text_spliter.split_documents(pages)

chunks = [c.page_content for c in text]
metadata = [c.metadata for c in text]


In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="./Hybrid_Data")

collection = client.get_or_create_collection(name="LLM_Halucinate",embedding_function=embedding_function)

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))], 
        metadatas=metadata
    )
'''so bm dosnt work with totaly string , so make it token before i put into it'''
from rank_bm25 import BM25Okapi
import string 
def fun(token):
    token = token.lower()
    trasn = str.maketrans('','',string.punctuation)
    token=token.translate(trasn)
    token = token.split()
    return token 
token_corpus = [fun(c) for c in chunks]
bm25  = BM25Okapi(token_corpus)
# print(collection.count())


In [ ]:
# Model load 
from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
Groq = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=key)


In [ ]:
def Hybrid_search(query:str):
    # dense 
    result = collection.query(query_texts=[query],n_results=5)
    document = result['documents'][0]
    distance = result['distances'][0]
    thresold = 1.0
    dense_filter=[doc for doc , d in zip(document ,distance) if thresold>d ]
    
    # sparse 
    score = bm25.get_scores(query.split())
    def get_top_index(score,k=10):
        indexed =  list(enumerate(score))
        indexed.sort(key=lambda x : x[1] , reverse= True)
        return [i for i , s in indexed[:10]]
    top_index=get_top_index(score,k=10)
    
    sparse_docs = [chunks[i] for i in top_index]
    
    # RPF marge 
    rrf_score = {}
    for rank , doc in enumerate(dense_filter):
        rrf_score[doc] = rrf_score.get(doc,0)+1/(60+rank)
    for rank , doc in enumerate(sparse_docs):
        rrf_score[doc] = rrf_score.get(doc,0)+1/(60+rank)
    merge = sorted(rrf_score.items(),key=lambda x:-x[1])
    top_docs = [doc for doc , _ in merge[:5]]
    
    if not top_docs:
        return "NOT RELEVENT CONTENT"
    return "/n/n".join(top_docs)
    


In [ ]:
qustion = "why model halucinate ?"
answer = Hybrid_search(qustion)
prompt = f"""provide answer based on the provided context only , dont use outside knolowedge
context : {answer},
qustion : {qustion}
"""
print("answers",Groq.invoke(prompt))


answers content='According to the provided context, a language model hallucinates because it is trained and graded in a way that rewards confident guessing over honest expression of uncertainty. This is due to prevailing benchmarks that score models primarily on accuracy, which means a confident guess is rewarded over an honest expression of uncertainty whenever the guess has any chance of being right.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 712, 'total_tokens': 780, 'completion_time': 0.108897419, 'completion_tokens_details': None, 'prompt_time': 0.048502076, 'prompt_tokens_details': None, 'queue_time': 0.050368433, 'total_time': 0.157399495}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--019f6182-a865-7892-baa1-864975bca892-0' usage_metadata={'input_tokens': 712, 'output_tokens': 68, 'total_tokens': 780}
